# 06 - Prepare RecBole Data

This notebook converts ReDial dialogue data to RecBole atomic file format for baseline comparison.

**Split strategy (mirrors LLM evaluation):**
- `user_liked` (preference signal) → `train.inter` for ALL users (warm-start)
- `recommended_accepted` (prediction targets) → `valid.inter` / `test.inter`
- This ensures RecBole models know each user's preferences (like LLMs get them in the prompt)
  and are evaluated on the same target: predicting accepted recommendations.

**Outputs:**
- `data/recbole/redial/redial.train.inter` - user_liked for all users + recommended_accepted for train users
- `data/recbole/redial/redial.valid.inter` - recommended_accepted for valid users
- `data/recbole/redial/redial.test.inter` - recommended_accepted for test users
- `data/recbole/redial/redial.item` - Item metadata
- `data/recbole/redial/evaluation_targets.json` - Ground truth targets for evaluation
- `data/recbole/redial/id_to_title.json` - Item ID to title mapping

In [ ]:
import json
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Configuration
DATA_PATH = Path("../data")
RECBOLE_DATA_PATH = DATA_PATH / "recbole" / "redial"
SEED = 42
VALID_SPLIT = 0.2  # 20% of train for validation

## Load Source Data

In [ ]:
# Load movie catalog
movies_df = pd.read_csv(DATA_PATH / "processed" / "movies_with_mentions_processed.csv")
print(f"Loaded {len(movies_df)} movies")
movies_df.head()

In [ ]:
# Load train dialogues
train_dialogues = []
with open(DATA_PATH / "processed" / "train_prompt_templates.jsonl") as f:
    for line in f:
        train_dialogues.append(json.loads(line))
print(f"Loaded {len(train_dialogues)} train dialogues")

# Load test dialogues
test_dialogues = []
with open(DATA_PATH / "processed" / "test_prompt_templates.jsonl") as f:
    for line in f:
        test_dialogues.append(json.loads(line))
print(f"Loaded {len(test_dialogues)} test dialogues")

In [ ]:
# Inspect a sample dialogue
sample = train_dialogues[0]
print(f"dialogue_id: {sample.get('dialogue_id')}")
print(f"user_liked: {sample.get('user_liked')}")
print(f"user_disliked: {sample.get('user_disliked')}")
print(f"recommended_accepted: {sample.get('recommended_accepted')}")
print(f"recommended_rejected: {sample.get('recommended_rejected')}")

## Create Title to ID Mapping

In [ ]:
# Create movie title to ID mapping (use both title and title_norm)
title_to_id = {}
for _, row in movies_df.iterrows():
    title_to_id[row["title"]] = row["id"]
    title_to_id[row["title_norm"]] = row["id"]

# Create ID to title mapping for later use
id_to_title = dict(zip(movies_df["id"], movies_df["title_norm"]))

print(f"Created mapping for {len(movies_df)} movies")
print(f"Total title variants: {len(title_to_id)}")

In [ ]:
def match_title_to_id(title: str, title_to_id: dict[str, int]) -> int | None:
    """Match a movie title to its ID with fuzzy matching fallback."""
    # Direct match
    if title in title_to_id:
        return title_to_id[title]

    # Normalized match (lowercase, stripped)
    title_lower = title.strip().lower()
    for key, val in title_to_id.items():
        if key.lower() == title_lower:
            return val

    return None  # No match found

## Convert Dialogues to Interactions

**Split logic:**
- `user_liked` (seen & liked movies) → training interactions for ALL users
- `recommended_accepted` (unseen, accepted recommendations) → evaluation targets
  - For train-only users: also goes into train.inter (extra training signal)
  - For valid users: goes into valid.inter
  - For test users: goes into test.inter

This mirrors LLM evaluation where `user_liked` is provided as context in the prompt
and `recommended_accepted` is the prediction target.

In [ ]:
def prepare_recbole_splits(
    train_dialogues: list[dict],
    valid_dialogues: list[dict],
    test_dialogues: list[dict],
    title_to_id: dict[str, int],
) -> dict:
    """Create RecBole interaction splits that mirror LLM evaluation.

    Split strategy:
      train.inter  = user_liked for ALL users + recommended_accepted for train users
      valid.inter  = recommended_accepted for valid users
      test.inter   = recommended_accepted for test users

    User IDs are assigned contiguously across all splits so that
    the same user appearing in train.inter and test.inter shares one ID.

    Returns dict with DataFrames and metadata for each split.
    """
    LIKED_TS_START = 0.0  # timestamps for user_liked items
    ACCEPTED_TS_START = 100.0  # timestamps for recommended_accepted items

    train_interactions = []  # user_liked (all) + recommended_accepted (train only)
    valid_interactions = []  # recommended_accepted (valid only)
    test_interactions = []  # recommended_accepted (test only)

    train_targets = {}
    valid_targets = {}
    test_targets = {}

    train_user_ids = []
    valid_user_ids = []
    test_user_ids = []

    unmatched_titles = set()
    skipped_no_history = 0
    skipped_no_targets = 0

    def _resolve_ids(titles):
        """Convert list of movie titles to item IDs."""
        ids = []
        for t in titles:
            item_id = match_title_to_id(t, title_to_id)
            if item_id is not None:
                ids.append(item_id)
            else:
                unmatched_titles.add(t)
        return ids

    user_id = 0  # global counter across all splits

    for split_name, dialogues in [
        ("train", train_dialogues),
        ("valid", valid_dialogues),
        ("test", test_dialogues),
    ]:
        for dialogue in tqdm(dialogues, desc=f"Processing {split_name}"):
            liked_ids = _resolve_ids(dialogue.get("user_liked", []))
            accepted_ids = _resolve_ids(dialogue.get("recommended_accepted", []))

            if not liked_ids:
                skipped_no_history += 1
                user_id += 1
                continue
            if not accepted_ids:
                skipped_no_targets += 1
                user_id += 1
                continue

            # user_liked → always goes to train.inter
            for ts, item_id in enumerate(liked_ids):
                train_interactions.append(
                    {
                        "user_id": user_id,
                        "item_id": item_id,
                        "timestamp": LIKED_TS_START + ts,
                    }
                )

            # recommended_accepted → goes to the split's .inter file
            target_rows = []
            for ts, item_id in enumerate(accepted_ids):
                target_rows.append(
                    {
                        "user_id": user_id,
                        "item_id": item_id,
                        "timestamp": ACCEPTED_TS_START + ts,
                    }
                )

            if split_name == "train":
                # Train users: recommended_accepted also in train.inter
                train_interactions.extend(target_rows)
                train_targets[user_id] = accepted_ids
                train_user_ids.append(user_id)
            elif split_name == "valid":
                valid_interactions.extend(target_rows)
                valid_targets[user_id] = accepted_ids
                valid_user_ids.append(user_id)
            else:  # test
                test_interactions.extend(target_rows)
                test_targets[user_id] = accepted_ids
                test_user_ids.append(user_id)

            user_id += 1

    print(f"Skipped {skipped_no_history} dialogues with no history")
    print(f"Skipped {skipped_no_targets} dialogues with no targets")
    print(f"Unmatched titles: {len(unmatched_titles)}")
    if unmatched_titles:
        print(f"Sample unmatched: {list(unmatched_titles)[:5]}")

    return {
        "train_df": pd.DataFrame(train_interactions),
        "valid_df": pd.DataFrame(valid_interactions),
        "test_df": pd.DataFrame(test_interactions),
        "train_targets": train_targets,
        "valid_targets": valid_targets,
        "test_targets": test_targets,
        "train_user_ids": train_user_ids,
        "valid_user_ids": valid_user_ids,
        "test_user_ids": test_user_ids,
    }

## Split Train Data into Train/Valid

In [ ]:
# Split original train dialogues into train/valid (80/20)
train_dialogues_split, valid_dialogues = train_test_split(
    train_dialogues,
    test_size=VALID_SPLIT,
    random_state=SEED,
)

print(f"Train dialogues: {len(train_dialogues_split)}")
print(f"Valid dialogues: {len(valid_dialogues)}")
print(f"Test dialogues: {len(test_dialogues)}")

## Process All Splits

Runs the new split logic:
- `user_liked` for ALL users → train.inter
- `recommended_accepted` for train users → train.inter (extra signal)
- `recommended_accepted` for valid users → valid.inter
- `recommended_accepted` for test users → test.inter

In [ ]:
splits = prepare_recbole_splits(
    train_dialogues_split, valid_dialogues, test_dialogues, title_to_id
)

train_inter_df = splits["train_df"]
valid_inter_df = splits["valid_df"]
test_inter_df = splits["test_df"]
train_targets = splits["train_targets"]
valid_targets = splits["valid_targets"]
test_targets = splits["test_targets"]
train_user_ids = splits["train_user_ids"]
valid_user_ids = splits["valid_user_ids"]
test_user_ids = splits["test_user_ids"]

print("\nSplit summary:")
print(
    f"  Train .inter: {len(train_inter_df)} interactions "
    f"(user_liked for all {len(train_user_ids) + len(valid_user_ids) + len(test_user_ids)} users "
    f"+ recommended_accepted for {len(train_user_ids)} train users)"
)
print(
    f"  Valid .inter: {len(valid_inter_df)} interactions "
    f"(recommended_accepted for {len(valid_user_ids)} valid users)"
)
print(
    f"  Test .inter:  {len(test_inter_df)} interactions "
    f"(recommended_accepted for {len(test_user_ids)} test users)"
)

## Create RecBole Atomic Files

In [ ]:
def create_inter_file(df: pd.DataFrame, output_path: Path) -> None:
    """Create RecBole .inter file with proper header format."""
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # RecBole header format: field_name:field_type
    header = "user_id:token\titem_id:token\ttimestamp:float"

    with open(output_path, "w") as f:
        f.write(header + "\n")
        for _, row in df.iterrows():
            f.write(
                f"{int(row['user_id'])}\t{int(row['item_id'])}\t{row['timestamp']}\n"
            )

    print(f"Saved {len(df)} interactions to {output_path}")

In [ ]:
# Create output directory
RECBOLE_DATA_PATH.mkdir(parents=True, exist_ok=True)

# Save .inter files for each split
create_inter_file(train_inter_df, RECBOLE_DATA_PATH / "redial.train.inter")
create_inter_file(valid_inter_df, RECBOLE_DATA_PATH / "redial.valid.inter")
create_inter_file(test_inter_df, RECBOLE_DATA_PATH / "redial.test.inter")

In [ ]:
def create_item_file(movies_df: pd.DataFrame, output_path: Path) -> None:
    """Create RecBole .item file with movie metadata."""
    # RecBole header format
    header = "item_id:token\ttitle:token_seq"

    with open(output_path, "w") as f:
        f.write(header + "\n")
        for _, row in movies_df.iterrows():
            # Escape special characters in title
            title = str(row["title_norm"]).replace("\t", " ").replace("\n", " ")
            f.write(f"{row['id']}\t{title}\n")

    print(f"Saved {len(movies_df)} items to {output_path}")


create_item_file(movies_df, RECBOLE_DATA_PATH / "redial.item")

## Save Evaluation Targets and Mappings

In [ ]:
# Save evaluation targets for all splits
targets_data = {
    "train_targets": {str(k): v for k, v in train_targets.items()},
    "valid_targets": {str(k): v for k, v in valid_targets.items()},
    "test_targets": {str(k): v for k, v in test_targets.items()},
    "train_user_ids": train_user_ids,
    "valid_user_ids": valid_user_ids,
    "test_user_ids": test_user_ids,
}

# Save dialogue_id → user_id mapping for content-based matching in evaluation notebooks.
# This is more robust than index-based matching (survives different skip conditions)
# and more reliable than ground-truth fingerprints (105 collisions in test set).
test_dialogue_to_user = {}
template_idx = 0
for split_name, dialogues in [
    ("train", train_dialogues_split),
    ("valid", valid_dialogues),
    ("test", test_dialogues),
]:
    for dialogue in dialogues:
        liked_ids = [
            match_title_to_id(t, title_to_id)
            for t in dialogue.get("user_liked", [])
            if match_title_to_id(t, title_to_id) is not None
        ]
        accepted_ids = [
            match_title_to_id(t, title_to_id)
            for t in dialogue.get("recommended_accepted", [])
            if match_title_to_id(t, title_to_id) is not None
        ]
        if not liked_ids or not accepted_ids:
            continue
        did = str(dialogue.get("dialogue_id", ""))
        if split_name == "test" and did:
            uid = test_user_ids[len(test_dialogue_to_user)]
            test_dialogue_to_user[did] = uid

targets_data["test_dialogue_to_user"] = test_dialogue_to_user
print(f"Built dialogue_id mapping for {len(test_dialogue_to_user)} test users")

with open(RECBOLE_DATA_PATH / "evaluation_targets.json", "w") as f:
    json.dump(targets_data, f, indent=2)

print("Saved evaluation targets:")
print(f"  Train: {len(train_targets)} users")
print(f"  Valid: {len(valid_targets)} users")
print(f"  Test: {len(test_targets)} users")

In [ ]:
# Save ID to title mapping
with open(RECBOLE_DATA_PATH / "id_to_title.json", "w") as f:
    json.dump({str(k): v for k, v in id_to_title.items()}, f, indent=2)

print(f"Saved {len(id_to_title)} item ID to title mappings")

## Summary Statistics

In [ ]:
print("\n" + "=" * 60)
print("RECBOLE DATA PREPARATION SUMMARY")
print("=" * 60)

total_users = len(train_user_ids) + len(valid_user_ids) + len(test_user_ids)
print(f"\nTotal users: {total_users}")
print(f"  Train-only: {len(train_user_ids)}")
print(f"  Valid:      {len(valid_user_ids)}")
print(f"  Test:       {len(test_user_ids)}")

print("\nInteraction counts:")
print(
    f"  train.inter: {len(train_inter_df)} (user_liked for all + recommended_accepted for train)"
)
print(f"  valid.inter: {len(valid_inter_df)} (recommended_accepted for valid)")
print(f"  test.inter:  {len(test_inter_df)} (recommended_accepted for test)")

print(
    f"\nUnique items across all splits: {pd.concat([train_inter_df, valid_inter_df, test_inter_df])['item_id'].nunique()}"
)

print(f"\nFiles created in {RECBOLE_DATA_PATH}:")
for f in sorted(RECBOLE_DATA_PATH.iterdir()):
    print(f"  - {f.name}")

In [ ]:
# Verify file format by reading first few lines
for fname in ["redial.train.inter", "redial.valid.inter", "redial.test.inter"]:
    print(f"\nSample from {fname}:")
    with open(RECBOLE_DATA_PATH / fname) as f:
        for i, line in enumerate(f):
            print(f"  {line.strip()}")
            if i >= 4:
                break

print("\nSample from redial.item:")
with open(RECBOLE_DATA_PATH / "redial.item") as f:
    for i, line in enumerate(f):
        print(f"  {line.strip()}")
        if i >= 3:
            break

In [ ]:
# Verify split properties
train_set = set(train_user_ids)
valid_set = set(valid_user_ids)
test_set = set(test_user_ids)

print("User split verification:")
print(f"  Train users: {len(train_set)}")
print(f"  Valid users: {len(valid_set)}")
print(f"  Test users:  {len(test_set)}")
print(f"  Train-Valid overlap: {len(train_set & valid_set)} (should be 0)")
print(f"  Train-Test overlap:  {len(train_set & test_set)} (should be 0)")
print(f"  Valid-Test overlap:  {len(valid_set & test_set)} (should be 0)")

# Verify train.inter contains ALL users (warm-start)
train_inter_users = set(train_inter_df["user_id"].unique())
all_eval_users = valid_set | test_set
missing_in_train = all_eval_users - train_inter_users
print("\nWarm-start verification:")
print(f"  Users in train.inter: {len(train_inter_users)}")
print(
    f"  Valid+Test users missing from train.inter: {len(missing_in_train)} (should be 0)"
)

# Verify no item leakage: for valid/test users, no item appears in both
# train.inter (user_liked) and valid/test.inter (recommended_accepted)
for split_name, split_df, split_users in [
    ("valid", valid_inter_df, valid_set),
    ("test", test_inter_df, test_set),
]:
    leaking_users = 0
    for uid in split_users:
        train_items = set(train_inter_df[train_inter_df["user_id"] == uid]["item_id"])
        eval_items = set(split_df[split_df["user_id"] == uid]["item_id"])
        overlap = train_items & eval_items
        if overlap:
            leaking_users += 1
    print(
        f"  {split_name} users with item overlap in train: {leaking_users} (should be 0)"
    )

In [ ]:
print("\nData preparation complete! Ready for RecBole HPO (notebook 07).")